
# 1. Environment Setup

This notebook performs data preprocessing, dataset validation, and experimental split generation for the BreakHis breast histopathology image classification project.

**Environment:**
- Operating System: Windows
- IDE: Visual Studio Code
- Python: 3.13.3
- GPU: NVIDIA RTX 4060 Laptop GPU
- Framework: PyTorch
- Dataset: BreakHis

**Classification task:** Binary classification of breast histopathology images into benign and malignant classes.


In [1]:

# ============================================
# 1.1 IMPORT REQUIRED LIBRARIES
# ============================================

# General utilities
import os
import re
import random
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter

# Image processing
from PIL import Image, ImageFile

# Visualization
import matplotlib.pyplot as plt

# Machine learning utilities
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold
)

# Deep learning framework
import torch
import torchvision

# Progress tracking
from tqdm.auto import tqdm

print("All libraries imported successfully!")


All libraries imported successfully!


In [2]:

# ============================================
# 1.2 SET RANDOM SEEDS
# ============================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)
print("Reproducibility settings initialized.")


Random seed: 42
Reproducibility settings initialized.


In [3]:

# ============================================
# 1.3 GPU AND ENVIRONMENT VERIFICATION
# ============================================

print("PyTorch version:", torch.__version__)
print("TorchVision version:", torchvision.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    device = torch.device("cuda")

    print("GPU detected:", torch.cuda.get_device_name(0))

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(f"GPU memory: {gpu_memory:.2f} GB")

else:

    device = torch.device("cpu")

    print("No CUDA GPU detected. Using CPU.")

print("\nSelected device:", device)


PyTorch version: 2.15.0+cu132
TorchVision version: 0.29.2+cu132

CUDA available: True
GPU detected: NVIDIA GeForce RTX 4060 Laptop GPU
GPU memory: 8.00 GB

Selected device: cuda


# 2. Project Paths and Dataset Verification

In this section, we define the locations of the original BreakHis dataset
and the directories where processed data and experiment outputs will be stored.

The original dataset will remain unchanged. All generated files will be
stored separately in the project directory.

All project paths are derived from `PROJECT_ROOT`. Image paths saved in the
manifest and split CSV files are relative to the project directory, so the
files can be reused on another machine by setting that machine's project root.

In [4]:
# SECTION 2: PROJECT PATHS

from pathlib import Path

# Keep the dataset folder structure inside the project directory.
DATASET_RELATIVE_PATH = (
    Path("archive (2)")
    / "BreaKHis_v1"
    / "BreaKHis_v1"
    / "histology_slides"
)

# Usually leave this as None.
# If automatic detection fails (for example, in Google Colab),
# set it to the full path of the project directory.
#
# Example for Google Drive:
# PROJECT_ROOT_OVERRIDE = Path("/content/drive/MyDrive/BreakHis_Project")
PROJECT_ROOT_OVERRIDE = None

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    # Search the current working directory and its parent directories.
    CURRENT_DIR = Path.cwd().resolve()
    candidate_roots = [CURRENT_DIR, *CURRENT_DIR.parents]

    PROJECT_ROOT = next(
        (
            candidate
            for candidate in candidate_roots
            if (candidate / DATASET_RELATIVE_PATH).is_dir()
        ),
        None
    )

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Could not automatically locate the project directory. "
            "Set PROJECT_ROOT_OVERRIDE to the project folder's path."
        )

# Dataset location
IMAGE_ROOT = PROJECT_ROOT / DATASET_RELATIVE_PATH

# Data directories
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

# Split directories
IMAGE_SPLITS_DIR = PROCESSED_DIR / "splits" / "image_level"
PATIENT_SPLITS_DIR = PROCESSED_DIR / "splits" / "patient_level"

# Output files
MANIFEST_PATH = PROCESSED_DIR / "breakhis_manifest.csv"
IMAGE_SPLITS_PATH = IMAGE_SPLITS_DIR / "image_level_folds.csv"
PATIENT_SPLITS_PATH = PATIENT_SPLITS_DIR / "patient_level_folds.csv"
ENVIRONMENT_PATH = PROCESSED_DIR / "environment_info.txt"

# Create output directories if they do not exist
IMAGE_SPLITS_DIR.mkdir(parents=True, exist_ok=True)
PATIENT_SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# Verify project paths
print("IMAGE ROOT:", IMAGE_ROOT.relative_to(PROJECT_ROOT))
print("PROCESSED DIRECTORY:", PROCESSED_DIR.relative_to(PROJECT_ROOT))

assert PROJECT_ROOT.is_dir(), "Project directory not found!"
assert IMAGE_ROOT.is_dir(), (
    f"Dataset directory not found: {IMAGE_ROOT}\n"
    "Check DATASET_RELATIVE_PATH and PROJECT_ROOT_OVERRIDE."
)

print("\nProject paths verified successfully.")

IMAGE ROOT: archive (2)\BreaKHis_v1\BreaKHis_v1\histology_slides
PROCESSED DIRECTORY: data\processed

Project paths verified successfully.


# 3. Raw Dataset Structure Inspection

In this section, we inspect the original BreakHis dataset before performing
any cleaning or preprocessing.

We examine:
- Binary classification folders (benign and malignant).
- Histological subtypes.
- Case and patient folder structure.
- Available magnifications.
- Number of image files in the raw dataset.

No images are removed, renamed, or modified in this section.

In [5]:

# Dataset's main classification directory
BREAST_DIR = IMAGE_ROOT / "breast"

class_folders = sorted(
    folder for folder in BREAST_DIR.iterdir()
    if folder.is_dir()
)

print("Classification folders:")

for folder in class_folders:
    print("-", folder.name)

print("\nNumber of classification folders:", len(class_folders))


Classification folders:
- benign
- malignant

Number of classification folders: 2


In [6]:

print("Histological subtypes:\n")

for class_folder in class_folders:

    # The dataset contains an intermediate SOB folder
    sob_dir = class_folder / "SOB"

    print(f"{class_folder.name.upper()}:")

    if not sob_dir.exists():
        print("  SOB folder not found!")
        continue

    subtype_folders = sorted(
        folder for folder in sob_dir.iterdir()
        if folder.is_dir()
    )

    for subtype in subtype_folders:
        print("  -", subtype.name)

    print("  Total subtypes:", len(subtype_folders))
    print()


Histological subtypes:

BENIGN:
  - adenosis
  - fibroadenoma
  - phyllodes_tumor
  - tubular_adenoma
  Total subtypes: 4

MALIGNANT:
  - ductal_carcinoma
  - lobular_carcinoma
  - mucinous_carcinoma
  - papillary_carcinoma
  Total subtypes: 4



In [ ]:

# Find the first image in the raw dataset
image_extensions = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

sample_images = sorted(
    path for path in IMAGE_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in image_extensions
)

if sample_images:
    sample_image = sample_images[0]

    print("Sample image path:")
    print(sample_image.relative_to(PROJECT_ROOT))

    print("\nFolder hierarchy:")
    for level, part in enumerate(sample_image.relative_to(IMAGE_ROOT).parts):
        print(f"Level {level}: {part}")
else:
    print("No image files found!")


Sample image path:
C:\Users\jaind\projects\CSD361 - Intro to ML Project\archive (2)\BreaKHis_v1\BreaKHis_v1\histology_slides\breast\benign\SOB\adenosis\SOB_B_A_14-22549AB\100X\SOB_B_A-14-22549AB-100-001.png

Folder hierarchy:
Level 0: breast
Level 1: benign
Level 2: SOB
Level 3: adenosis
Level 4: SOB_B_A_14-22549AB
Level 5: 100X
Level 6: SOB_B_A-14-22549AB-100-001.png


In [8]:

# Collect information about every raw image
raw_records = []

for image_path in sample_images:

    relative_parts = image_path.relative_to(IMAGE_ROOT).parts

    # Actual structure:
    # breast / class / SOB / subtype / case_id / magnification / filename
    if len(relative_parts) != 7:
        continue

    _, class_name, sob, subtype, case_id, magnification, filename = relative_parts

    raw_records.append({
        "image_path": str(image_path),
        "class": class_name,
        "subtype": subtype,
        "case_id": case_id,
        "magnification": magnification,
        "filename": filename
    })

raw_df = pd.DataFrame(raw_records)

print("Raw dataset summary")
print("-------------------")
print("Total image files:", len(raw_df))
print("Unique case folders:", raw_df["case_id"].nunique())

print("\nImages by class:")
print(raw_df["class"].value_counts().sort_index())

print("\nImages by subtype:")
print(raw_df["subtype"].value_counts().sort_index())

print("\nImages by magnification:")
print(raw_df["magnification"].value_counts().sort_index())


Raw dataset summary
-------------------
Total image files: 7909
Unique case folders: 82

Images by class:
class
benign       2480
malignant    5429
Name: count, dtype: int64

Images by subtype:
subtype
adenosis                444
ductal_carcinoma       3451
fibroadenoma           1014
lobular_carcinoma       626
mucinous_carcinoma      792
papillary_carcinoma     560
phyllodes_tumor         453
tubular_adenoma         569
Name: count, dtype: int64

Images by magnification:
magnification
100X    2081
200X    2013
400X    1820
40X     1995
Name: count, dtype: int64


In [9]:

class_magnification_table = pd.crosstab(
    raw_df["class"],
    raw_df["magnification"],
    margins=True
)

print("Image distribution by class and magnification:")
display(class_magnification_table)


Image distribution by class and magnification:


magnification,100X,200X,400X,40X,All
class,,,,,
benign,644,623,588,625,2480
malignant,1437,1390,1232,1370,5429
All,2081,2013,1820,1995,7909


In [10]:

# Extract candidate patient identifiers from case folder names
raw_df["candidate_patient_id"] = raw_df["case_id"].str.extract(
    r"(\d+-\d+[A-Z]*)",
    expand=False
)

# Find candidate IDs associated with multiple case folders
patient_case_counts = (
    raw_df.groupby("candidate_patient_id")["case_id"]
    .nunique()
    .sort_values(ascending=False)
)

repeated_candidate_ids = patient_case_counts[
    patient_case_counts > 1
]

print("Candidate patient IDs associated with multiple case folders:")

if repeated_candidate_ids.empty:
    print("No repeated candidate IDs found.")
else:
    print(repeated_candidate_ids)

    print("\nDetails:")

    for patient_id in repeated_candidate_ids.index:
        matching_rows = raw_df[
            raw_df["candidate_patient_id"] == patient_id
        ]

        print(f"\nCandidate ID: {patient_id}")
        print("Case folders:", sorted(matching_rows["case_id"].unique()))
        print("Subtypes:", sorted(matching_rows["subtype"].unique()))
        print("Unique images:", matching_rows["image_path"].nunique())


Candidate patient IDs associated with multiple case folders:
candidate_patient_id
14-13412    2
Name: case_id, dtype: int64

Details:

Candidate ID: 14-13412
Case folders: ['SOB_M_DC_14-13412', 'SOB_M_LC_14-13412']
Subtypes: ['ductal_carcinoma', 'lobular_carcinoma']
Unique images: 246


# 4. Dataset Cleaning and Manifest Creation

The raw BreakHis dataset contains 7,909 images across 82 case folders.

Two case folders share the candidate patient identifier 14-13412:
- SOB_M_DC_14-13412
- SOB_M_LC_14-13412

Since their biological patient identity cannot be independently established,
both folders are conservatively excluded from the processed dataset.

This removes 246 unique images.

The original dataset is preserved. Cleaning is performed by filtering
the dataset manifest rather than deleting or modifying image files.

Binary labels:
- Benign = 0
- Malignant = 1

In [11]:

# Case folders excluded due to ambiguous patient identity
EXCLUDED_CASES = {
    "SOB_M_DC_14-13412",
    "SOB_M_LC_14-13412"
}

# Create a cleaned copy without modifying raw_df
clean_df = raw_df[
    ~raw_df["case_id"].isin(EXCLUDED_CASES)
].copy()

# Assign binary classification labels
clean_df["label"] = clean_df["class"].map({
    "benign": 0,
    "malignant": 1
})

# Extract candidate patient group ID, preserving suffix letters
clean_df["patient_group_id"] = clean_df["case_id"].str.extract(
    r"(\d+-\d+[A-Z]*)",
    expand=False
)

# Convert image paths to project-relative paths.
# This makes the manifest portable across Windows, Linux, and Google Colab.
clean_df["image_path"] = clean_df["image_path"].apply(
    lambda path: Path(path).resolve()
    .relative_to(PROJECT_ROOT.resolve())
    .as_posix()
)

# Reset row numbering
clean_df = clean_df.reset_index(drop=True)

print("Cleaning completed.")
print("Raw images:", len(raw_df))
print("Excluded images:", len(raw_df) - len(clean_df))
print("Remaining images:", len(clean_df))
print("Remaining case folders:", clean_df["case_id"].nunique())
print("Candidate patient groups:", clean_df["patient_group_id"].nunique())


Cleaning completed.
Raw images: 7909
Excluded images: 246
Remaining images: 7663
Remaining case folders: 80
Candidate patient groups: 80


In [12]:

print("CLEANED DATASET SUMMARY")
print("-----------------------")

print("\nImages by class:")
print(clean_df["class"].value_counts().sort_index())

print("\nImages by subtype:")
print(clean_df["subtype"].value_counts().sort_index())

print("\nImages by magnification:")
print(clean_df["magnification"].value_counts().sort_index())

print("\nClass distribution by magnification:")
display(
    pd.crosstab(
        clean_df["class"],
        clean_df["magnification"],
        margins=True
    )
)


CLEANED DATASET SUMMARY
-----------------------

Images by class:
class
benign       2480
malignant    5183
Name: count, dtype: int64

Images by subtype:
subtype
adenosis                444
ductal_carcinoma       3328
fibroadenoma           1014
lobular_carcinoma       503
mucinous_carcinoma      792
papillary_carcinoma     560
phyllodes_tumor         453
tubular_adenoma         569
Name: count, dtype: int64

Images by magnification:
magnification
100X    2015
200X    1949
400X    1768
40X     1931
Name: count, dtype: int64

Class distribution by magnification:


magnification,100X,200X,400X,40X,All
class,,,,,
benign,644,623,588,625,2480
malignant,1371,1326,1180,1306,5183
All,2015,1949,1768,1931,7663


In [13]:

print("MANIFEST VALIDATION")
print("-------------------")

print("Total rows:", len(clean_df))
print("\nMissing values:")
print(clean_df.isnull().sum())

duplicate_paths = clean_df["image_path"].duplicated().sum()
duplicate_filenames = clean_df["filename"].duplicated().sum()

missing_files = sum(
    not (PROJECT_ROOT / p).is_file()
    for p in clean_df["image_path"]
)

unexpected_labels = (
    set(clean_df["label"].dropna()) - {0, 1}
)

excluded_cases_present = (
    clean_df["case_id"].isin(EXCLUDED_CASES).sum()
)

missing_values = clean_df.isnull().sum().sum()

print("\nDuplicate image paths:", duplicate_paths)
print("Duplicate filenames:", duplicate_filenames)
print("Missing image files:", missing_files)
print("Unexpected labels:", unexpected_labels)
print("Excluded case folders still present:", excluded_cases_present)

# Automated validation
assert len(clean_df) == 7663
assert missing_values == 0
assert duplicate_paths == 0
assert duplicate_filenames == 0
assert missing_files == 0
assert len(unexpected_labels) == 0
assert excluded_cases_present == 0

print("\nPASS: All manifest validation checks passed.")


MANIFEST VALIDATION
-------------------
Total rows: 7663

Missing values:
image_path              0
class                   0
subtype                 0
case_id                 0
magnification           0
filename                0
candidate_patient_id    0
label                   0
patient_group_id        0
dtype: int64

Duplicate image paths: 0
Duplicate filenames: 0
Missing image files: 0
Unexpected labels: set()
Excluded case folders still present: 0

PASS: All manifest validation checks passed.


In [14]:

from tqdm.auto import tqdm

corrupt_images = []
image_dimensions = []

for image_path in tqdm(clean_df["image_path"], desc="Checking images"):

    try:
        resolved_image_path = PROJECT_ROOT / image_path

        with Image.open(resolved_image_path) as img:
            img.verify()

        # Reopen after verify to retrieve dimensions
        with Image.open(resolved_image_path) as img:
            image_dimensions.append(img.size)

    except Exception as error:
        corrupt_images.append({
            "image_path": image_path,
            "error": str(error)
        })
        
assert len(image_dimensions) == len(clean_df)
assert len(corrupt_images) == 0

print("PASS: All images are readable.")        

print("\nIMAGE INTEGRITY RESULTS")
print("-----------------------")
print("Images checked:", len(clean_df))
print("Readable images:", len(image_dimensions))
print("Corrupt/unreadable images:", len(corrupt_images))

if corrupt_images:
    display(pd.DataFrame(corrupt_images))


Checking images:   0%|          | 0/7663 [00:00<?, ?it/s]

PASS: All images are readable.

IMAGE INTEGRITY RESULTS
-----------------------
Images checked: 7663
Readable images: 7663
Corrupt/unreadable images: 0


In [15]:

# Keep only the columns needed for the preprocessing pipeline
manifest_columns = [
    "image_path",
    "filename",
    "label",
    "class",
    "subtype",
    "case_id",
    "patient_group_id",
    "magnification"
]

manifest_df = clean_df[manifest_columns].copy()

# Save the processed manifest
manifest_df.to_csv(MANIFEST_PATH, index=False)

print("Manifest saved successfully!")
print("Location:", MANIFEST_PATH.relative_to(PROJECT_ROOT))
print("Rows:", len(manifest_df))
print("Columns:", list(manifest_df.columns))


Manifest saved successfully!
Location: data\processed\breakhis_manifest.csv
Rows: 7663
Columns: ['image_path', 'filename', 'label', 'class', 'subtype', 'case_id', 'patient_group_id', 'magnification']



# 5. Cross-Validation Split Preparation

In this section, we prepare two 5-fold cross-validation protocols to study model generalization:

1. **Image-level evaluation:** Individual images are split into five folds, allowing images from the same candidate patient group to appear in both training and testing sets.
2. **Patient-level evaluation:** Images are grouped by candidate patient ID to ensure that no candidate patient group appears in both training and testing sets.

The first experiment compares these protocols using all magnifications together to investigate generalization to unseen patients.

The second experiment investigates the effect of magnification by evaluating performance separately at 40X, 100X, 200X and 400X.

All fold assignments are validated and saved for use in feature extraction and model evaluation.


In [16]:

# Load the cleaned dataset manifest
manifest_df = pd.read_csv(MANIFEST_PATH)

print("Manifest loaded successfully!")
print("Dataset shape:", manifest_df.shape)

print("\nTotal images:", len(manifest_df))
print("Candidate patient groups:", manifest_df["patient_group_id"].nunique())

print("\nClass distribution:")
print(manifest_df["class"].value_counts())

print("\nMagnification distribution:")
print(manifest_df["magnification"].value_counts().sort_index())

print("\nMissing values:")
print(manifest_df.isnull().sum().sum())


Manifest loaded successfully!
Dataset shape: (7663, 8)

Total images: 7663
Candidate patient groups: 80

Class distribution:
class
malignant    5183
benign       2480
Name: count, dtype: int64

Magnification distribution:
magnification
100X    2015
200X    1949
400X    1768
40X     1931
Name: count, dtype: int64

Missing values:
0


In [17]:

# Check whether each candidate patient group has a consistent label
patient_label_counts = (
    manifest_df.groupby("patient_group_id")["label"]
    .nunique()
)

inconsistent_patients = patient_label_counts[
    patient_label_counts > 1
]

print("Candidate patient groups:", len(patient_label_counts))
print("Groups with inconsistent labels:", len(inconsistent_patients))

if inconsistent_patients.empty:
    print("PASS: All candidate patient groups have consistent labels.")
else:
    print("ERROR: Some groups have multiple labels.")
    print(inconsistent_patients)
    
assert inconsistent_patients.empty, (
    "Some candidate patient groups have inconsistent labels!"
)


Candidate patient groups: 80
Groups with inconsistent labels: 0
PASS: All candidate patient groups have consistent labels.


In [18]:

# Initialize image-level cross-validation
image_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

# Create a copy of the complete manifest
image_splits = manifest_df.copy()

# Initialize fold assignments
image_splits["fold_id"] = 0

# Generate five folds across the entire dataset
for fold_number, (_, test_indices) in enumerate(
    image_cv.split(
        X=image_splits,
        y=image_splits["label"]
    ),
    start=1
):
    image_splits.loc[test_indices, "fold_id"] = fold_number

print("Image-level folds generated successfully!")

print("\nImages assigned to each fold:")
print(image_splits["fold_id"].value_counts().sort_index())

print("\nClass distribution by fold:")
display(
    pd.crosstab(
        image_splits["fold_id"],
        image_splits["class"],
        margins=True
    )
)


Image-level folds generated successfully!

Images assigned to each fold:
fold_id
1    1533
2    1533
3    1533
4    1532
5    1532
Name: count, dtype: int64

Class distribution by fold:


class,benign,malignant,All
fold_id,,,
1,496,1037,1533
2,496,1037,1533
3,496,1037,1533
4,496,1036,1532
5,496,1036,1532
All,2480,5183,7663


In [19]:

# Create one row per candidate patient group
patient_df = (
    manifest_df[["patient_group_id", "label"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Initialize patient-level stratified group cross-validation
patient_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

# Assign one fold to each candidate patient group
patient_df["fold_id"] = 0

for fold_number, (_, test_indices) in enumerate(
    patient_cv.split(
        X=np.zeros(len(patient_df)),
        y=patient_df["label"],
        groups=patient_df["patient_group_id"]
    ),
    start=1
):
    patient_df.loc[test_indices, "fold_id"] = fold_number

# Create a mapping between candidate patient groups and fold IDs
patient_fold_mapping = patient_df.set_index(
    "patient_group_id"
)["fold_id"]

# Apply the same fold assignment to every image
patient_splits = manifest_df.copy()

patient_splits["fold_id"] = (
    patient_splits["patient_group_id"]
    .map(patient_fold_mapping)
)

# Display fold distribution
print("Patient-level folds generated successfully!")

print("\nCandidate patient groups per fold:")
print(
    patient_df["fold_id"]
    .value_counts()
    .sort_index()
)

print("\nImage distribution per fold:")
print(
    patient_splits["fold_id"]
    .value_counts()
    .sort_index()
)

print("\nClass distribution by fold:")
display(
    pd.crosstab(
        patient_splits["fold_id"],
        patient_splits["class"],
        margins=True
    )
)



Patient-level folds generated successfully!

Candidate patient groups per fold:
fold_id
1    16
2    16
3    16
4    16
5    16
Name: count, dtype: int64

Image distribution per fold:
fold_id
1    1627
2    1560
3    1649
4    1414
5    1413
Name: count, dtype: int64

Class distribution by fold:


class,benign,malignant,All
fold_id,,,
1,609,1018,1627
2,338,1222,1560
3,622,1027,1649
4,480,934,1414
5,431,982,1413
All,2480,5183,7663


In [20]:

def validate_folds(split_df, protocol_name):

    print(f"\n{'=' * 55}")
    print(f"{protocol_name.upper()} VALIDATION")
    print("=" * 55)

    for fold in range(1, 6):

        test_data = split_df[
            split_df["fold_id"] == fold
        ]

        train_data = split_df[
            split_df["fold_id"] != fold
        ]

        train_groups = set(train_data["patient_group_id"])
        test_groups = set(test_data["patient_group_id"])

        overlap = train_groups.intersection(test_groups)

        print(f"\nFold {fold}")
        print("-" * 35)

        print("Training images:", len(train_data))
        print("Testing images:", len(test_data))

        print("Training class counts:")
        print(train_data["class"].value_counts().to_dict())

        print("Testing class counts:")
        print(test_data["class"].value_counts().to_dict())

        print("Candidate patient overlap:", len(overlap))

        if protocol_name == "Patient-level":
            assert len(overlap) == 0, (
                f"Patient overlap detected in fold {fold}!"
            )
            print("PASS: No candidate patient overlap.")

        else:
            print("Image-level protocol: patient overlap is permitted.")


# Validate both protocols
validate_folds(image_splits, "Image-level")
validate_folds(patient_splits, "Patient-level")



IMAGE-LEVEL VALIDATION

Fold 1
-----------------------------------
Training images: 6130
Testing images: 1533
Training class counts:
{'malignant': 4146, 'benign': 1984}
Testing class counts:
{'malignant': 1037, 'benign': 496}
Candidate patient overlap: 80
Image-level protocol: patient overlap is permitted.

Fold 2
-----------------------------------
Training images: 6130
Testing images: 1533
Training class counts:
{'malignant': 4146, 'benign': 1984}
Testing class counts:
{'malignant': 1037, 'benign': 496}
Candidate patient overlap: 80
Image-level protocol: patient overlap is permitted.

Fold 3
-----------------------------------
Training images: 6130
Testing images: 1533
Training class counts:
{'malignant': 4146, 'benign': 1984}
Testing class counts:
{'malignant': 1037, 'benign': 496}
Candidate patient overlap: 80
Image-level protocol: patient overlap is permitted.

Fold 4
-----------------------------------
Training images: 6131
Testing images: 1532
Training class counts:
{'malignant

In [21]:

magnifications = ["40X", "100X", "200X", "400X"]

print("MAGNIFICATION-WISE PATIENT-LEVEL EVALUATION")
print("=" * 55)

for magnification in magnifications:

    # Filter the complete patient-level split
    mag_data = patient_splits[
        patient_splits["magnification"] == magnification
    ]

    print(f"\nMagnification: {magnification}")
    print("Total images:", len(mag_data))

    for fold in range(1, 6):

        # Training and testing data for this magnification
        test_data = mag_data[
            mag_data["fold_id"] == fold
        ]

        train_data = mag_data[
            mag_data["fold_id"] != fold
        ]
        
        # Validate patient-level separation
        assert not set(train_data["patient_group_id"]).intersection(
            set(test_data["patient_group_id"])
        )
        
        # Ensure both classes are present in the test set
        assert set(test_data["label"].unique()) == {0, 1}

        print(
            f"Fold {fold}: "
            f"Train = {len(train_data)}, "
            f"Test = {len(test_data)}, "
            f"Test benign = {(test_data['label'] == 0).sum()}, "
            f"Test malignant = {(test_data['label'] == 1).sum()}"
        )


MAGNIFICATION-WISE PATIENT-LEVEL EVALUATION

Magnification: 40X
Total images: 1931
Fold 1: Train = 1514, Test = 417, Test benign = 154, Test malignant = 263
Fold 2: Train = 1552, Test = 379, Test benign = 81, Test malignant = 298
Fold 3: Train = 1509, Test = 422, Test benign = 162, Test malignant = 260
Fold 4: Train = 1576, Test = 355, Test benign = 119, Test malignant = 236
Fold 5: Train = 1573, Test = 358, Test benign = 109, Test malignant = 249

Magnification: 100X
Total images: 2015
Fold 1: Train = 1579, Test = 436, Test benign = 159, Test malignant = 277
Fold 2: Train = 1614, Test = 401, Test benign = 93, Test malignant = 308
Fold 3: Train = 1580, Test = 435, Test benign = 156, Test malignant = 279
Fold 4: Train = 1631, Test = 384, Test benign = 126, Test malignant = 258
Fold 5: Train = 1656, Test = 359, Test benign = 110, Test malignant = 249

Magnification: 200X
Total images: 1949
Fold 1: Train = 1554, Test = 395, Test benign = 149, Test malignant = 246
Fold 2: Train = 1531, Tes

In [22]:

print("FINAL SPLIT VALIDATION")
print("=" * 45)

# Check image-level assignments
print("\nImage-level:")
print("Missing assignments:", image_splits["fold_id"].isna().sum())
print("Invalid fold IDs:", (~image_splits["fold_id"].isin(range(1, 6))).sum())

# Check patient-level assignments
print("\nPatient-level:")
print("Missing assignments:", patient_splits["fold_id"].isna().sum())
print("Invalid fold IDs:", (~patient_splits["fold_id"].isin(range(1, 6))).sum())

# Check candidate patient group consistency
patient_fold_counts = (
    patient_splits.groupby("patient_group_id")["fold_id"].nunique()
)

print("\nGroups assigned to multiple folds:",
      (patient_fold_counts > 1).sum())

# Check patient-level train/test overlap
leakage_detected = False

for fold in range(1, 6):

    train_groups = set(
        patient_splits.loc[
            patient_splits["fold_id"] != fold,
            "patient_group_id"
        ]
    )

    test_groups = set(
        patient_splits.loc[
            patient_splits["fold_id"] == fold,
            "patient_group_id"
        ]
    )

    if train_groups.intersection(test_groups):
        leakage_detected = True

print("Patient-level leakage detected:", leakage_detected)

assert image_splits["fold_id"].isin(range(1, 6)).all()
assert patient_splits["fold_id"].isin(range(1, 6)).all()
assert patient_fold_counts.eq(1).all()
assert not leakage_detected

print("\nAll final validation checks passed!")



FINAL SPLIT VALIDATION

Image-level:
Missing assignments: 0
Invalid fold IDs: 0

Patient-level:
Missing assignments: 0
Invalid fold IDs: 0

Groups assigned to multiple folds: 0
Patient-level leakage detected: False

All final validation checks passed!


In [23]:

# Save the complete split assignments
image_splits.to_csv(IMAGE_SPLITS_PATH, index=False)
patient_splits.to_csv(PATIENT_SPLITS_PATH, index=False)

print("Cross-validation split files saved successfully!")

print("\nImage-level split:")
print(IMAGE_SPLITS_PATH.relative_to(PROJECT_ROOT))

print("\nPatient-level split:")
print(PATIENT_SPLITS_PATH.relative_to(PROJECT_ROOT))


Cross-validation split files saved successfully!

Image-level split:
data\processed\splits\image_level\image_level_folds.csv

Patient-level split:
data\processed\splits\patient_level\patient_level_folds.csv


In [24]:

# Count the number of different folds assigned to each candidate patient group
patient_fold_counts = (
    patient_splits.groupby("patient_group_id")["fold_id"]
    .nunique()
)

inconsistent_groups = patient_fold_counts[
    patient_fold_counts > 1
]

print("Candidate patient groups:", len(patient_fold_counts))
print("Groups assigned to multiple folds:", len(inconsistent_groups))

if inconsistent_groups.empty:
    print("\nPASS: Every candidate patient group has one consistent fold.")
else:
    print("\nERROR: Inconsistent fold assignments detected.")
    print(inconsistent_groups)


Candidate patient groups: 80
Groups assigned to multiple folds: 0

PASS: Every candidate patient group has one consistent fold.



# 6. Final Validation and Reproducibility

In this section, we perform the final checks on the generated
cross-validation splits and saved dataset files.

We verify fold distributions, confirm the integrity of saved
artifacts, and record the software environment for reproducibility.

The validated manifest and split files will serve as inputs
for the subsequent feature extraction and model training stages.


In [25]:

# Count images of each magnification in every image-level fold
image_mag_counts = pd.crosstab(
    image_splits["fold_id"],
    image_splits["magnification"]
)

print("Image-level fold distribution by magnification:")
display(image_mag_counts)

# Compare magnification proportions within each fold
image_mag_percentages = (
    image_mag_counts
    .div(image_mag_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

print("\nMagnification percentages within each fold:")
display(image_mag_percentages)

# Overall magnification proportions
overall_mag_percentages = (
    manifest_df["magnification"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .sort_index()
)

print("\nOverall magnification percentages:")
display(overall_mag_percentages)


Image-level fold distribution by magnification:


magnification,100X,200X,400X,40X
fold_id,,,,
1,409,397,351,376
2,393,385,330,425
3,410,392,365,366
4,415,390,365,362
5,388,385,357,402



Magnification percentages within each fold:


magnification,100X,200X,400X,40X
fold_id,,,,
1,26.68,25.90,22.90,24.53
2,25.64,25.11,21.53,27.72
3,26.74,25.57,23.81,23.87
4,27.09,25.46,23.83,23.63
5,25.33,25.13,23.30,26.24



Overall magnification percentages:


magnification
100X    26.30
200X    25.43
400X    23.07
40X     25.20
Name: proportion, dtype: float64

In [26]:

# Check that all three important output files exist
required_files = {
    "Cleaned manifest": MANIFEST_PATH,
    "Image-level folds": IMAGE_SPLITS_PATH,
    "Patient-level folds": PATIENT_SPLITS_PATH
}

for name, path in required_files.items():
    assert path.is_file(), f"{name} is missing: {path}"

print("All required output files exist.")

print("SAVED FILE VERIFICATION")
print("=" * 45)

for name, path in required_files.items():
    print(f"{name}: {'FOUND' if path.is_file() else 'MISSING'}")
    print("Path:", path.relative_to(PROJECT_ROOT))

# Reload the saved files
saved_manifest = pd.read_csv(MANIFEST_PATH)
saved_image_splits = pd.read_csv(IMAGE_SPLITS_PATH)
saved_patient_splits = pd.read_csv(PATIENT_SPLITS_PATH)

# Verify row counts
assert len(saved_manifest) == 7663
assert len(saved_image_splits) == 7663
assert len(saved_patient_splits) == 7663

# Verify that image paths are portable project-relative paths
assert not saved_manifest["image_path"].map(
    lambda path: Path(path).is_absolute()
).any(), "Absolute image paths found in the manifest!"

# Verify that a sample image can be resolved from the project root
sample_relative_path = saved_manifest["image_path"].iloc[0]
sample_resolved_path = PROJECT_ROOT / sample_relative_path
assert sample_resolved_path.is_file(), (
    f"Could not resolve sample image: {sample_resolved_path}"
)

# Verify that both split files preserve the manifest's image ordering
assert saved_image_splits["image_path"].equals(
    saved_manifest["image_path"]
)
assert saved_patient_splits["image_path"].equals(
    saved_manifest["image_path"]
)

# Verify fold assignments
assert saved_image_splits["fold_id"].isin(range(1, 6)).all()
assert saved_patient_splits["fold_id"].isin(range(1, 6)).all()

# Verify patient-level separation
for fold in range(1, 6):

    train_groups = set(
        saved_patient_splits.loc[
            saved_patient_splits["fold_id"] != fold,
            "patient_group_id"
        ]
    )

    test_groups = set(
        saved_patient_splits.loc[
            saved_patient_splits["fold_id"] == fold,
            "patient_group_id"
        ]
    )

    assert train_groups.isdisjoint(test_groups)

print("\nAll saved files reloaded successfully.")
print("All row counts and fold assignments verified.")
print("Patient-level leakage check passed for all five folds.")


All required output files exist.
SAVED FILE VERIFICATION
Cleaned manifest: FOUND
Path: data\processed\breakhis_manifest.csv
Image-level folds: FOUND
Path: data\processed\splits\image_level\image_level_folds.csv
Patient-level folds: FOUND
Path: data\processed\splits\patient_level\patient_level_folds.csv

All saved files reloaded successfully.
All row counts and fold assignments verified.
Patient-level leakage check passed for all five folds.


In [27]:

import sys
import platform
import sklearn
import PIL

environment_info = {
    "Python": sys.version.split()[0],
    "Operating System": platform.platform(),
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
    "Pillow": PIL.__version__,
    "Scikit-learn": sklearn.__version__,
    "PyTorch": torch.__version__,
    "TorchVision": torchvision.__version__,
    "CUDA available": str(torch.cuda.is_available()),
    "GPU": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "Not available"
    )
}

print("PROJECT ENVIRONMENT")
print("=" * 45)

for package, version in environment_info.items():
    print(f"{package}: {version}")

# Save environment information
with open(ENVIRONMENT_PATH, "w", encoding="utf-8") as file:
    for package, version in environment_info.items():
        file.write(f"{package}: {version}\n")

print("\nEnvironment information saved to:")
print(ENVIRONMENT_PATH.relative_to(PROJECT_ROOT))


PROJECT ENVIRONMENT
Python: 3.13.3
Operating System: Windows-11-10.0.26200-SP0
NumPy: 2.5.4
Pandas: 3.1.1
Pillow: 13.0.0
Scikit-learn: 1.9.1
PyTorch: 2.15.0+cu132
TorchVision: 0.29.2+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

Environment information saved to:
data\processed\environment_info.txt
